# Install Packages

In [ ]:
!pip install demucs
!pip install demucs torch torchaudio
!pip install torchcodec
!pip install -U openai-whisper

  Using cached demucs-4.0.1.tar.gz (1.2 MB)
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.1/87.1 kB 6.3 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.6/59.6 kB 2.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 249.1/249.1 kB 18.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.0/40.0 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.0/76.0 kB 4.9 MB/s eta 0:00:00
  Created wheel for demucs: filename=demucs-4.0.1-py3-none-any.whl size=78388 sha256=213813a1c4e2e069a005851819d0db3bf8d362a0e93949fcd7ce3d2220281c9e
  Stored in directory: /root/.cache/pip/wheels/1b/0c/20/a3b3daa1f9b65c8b0445729f94740ec335d0f86f1066c5c414
  Created wheel for julius: filename=julius-0.2.7-py3-none-any.whl size=21870 sha256=

# Vocal and Music Seperation

In [ ]:
!demucs -h  #Help


usage: demucs.separate [-h] [-s SIG | -n NAME] [--repo REPO] [-v] [-o OUT]
                       [--filename FILENAME] [-d DEVICE] [--shifts SHIFTS]
                       [--overlap OVERLAP] [--no-split | --segment SEGMENT]
                       [--two-stems STEM] [--int24 | --float32]
                       [--clip-mode {rescale,clamp}] [--flac | --mp3]
                       [--mp3-bitrate MP3_BITRATE]
                       [--mp3-preset {2,3,4,5,6,7}] [-j JOBS]
                       tracks [tracks ...]

Separate the sources for the given tracks

positional arguments:
  tracks                Path to tracks

options:
  -h, --help            show this help message and exit
  -s SIG, --sig SIG     Locally trained XP signature.
  -n NAME, --name NAME  Pretrained model name or signature. Default is
                        htdemucs.
  --repo REPO           Folder containing all pre-trained models for use with
                        -n.
  -v, --verbose
  -o OUT, --out OUT     Folder w

In [ ]:
!python -m demucs.separate -n htdemucs_ft -d cuda --two-stems vocals /content/Edd_Sheeran.wav


Downloading: "https://dl.fbaipublicfiles.com/demucs/hybrid_transformer/f7e0c4bc-ba3fe64a.th" to /root/.cache/torch/hub/checkpoints/f7e0c4bc-ba3fe64a.th
100% 80.2M/80.2M [00:00<00:00, 157MB/s]
Downloading: "https://dl.fbaipublicfiles.com/demucs/hybrid_transformer/d12395a8-e57c48e6.th" to /root/.cache/torch/hub/checkpoints/d12395a8-e57c48e6.th
100% 80.2M/80.2M [00:00<00:00, 133MB/s]
Downloading: "https://dl.fbaipublicfiles.com/demucs/hybrid_transformer/92cfc3b6-ef3bcb9c.th" to /root/.cache/torch/hub/checkpoints/92cfc3b6-ef3bcb9c.th
100% 80.2M/80.2M [00:00<00:00, 163MB/s]
Downloading: "https://dl.fbaipublicfiles.com/demucs/hybrid_transformer/04573f0d-f3cf25b2.th" to /root/.cache/torch/hub/checkpoints/04573f0d-f3cf25b2.th
100% 80.2M/80.2M [00:00<00:00, 137MB/s]
Selected model is a bag of 4 models. You will see that many progress bars per track.
Separated tracks will be stored in /content/separated/htdemucs_ft
Separating track /content/Edd_Sheeran.wav
100%|██████████████████████████████████

#ASR

In [ ]:
import whisper

model = whisper.load_model("large-v3")

result = model.transcribe("/content/separated/htdemucs_ft/Edd_Sheeran/vocals.wav")

for seg in result["segments"]:
    print(f"[{seg['start']:.2f} → {seg['end']:.2f}] {seg['text']}")


[0.00 → 3.00]  Subtitles by the Amara.org community
[12.00 → 17.00]  The club isn't the best place to find a lover, so the bar is where I go
[17.00 → 22.00]  Me and my friends at the table doing shots, drinking fast and then we talk slow
[22.00 → 27.00]  Come over and start up a conversation with just me and trust me I'll give it a chance
[27.00 → 31.00]  Take my hand, stop it, find the man on the jukebox and then we start to dance
[31.00 → 34.00]  And I'm singing like girl you know I want your love
[34.00 → 37.00]  Your love was handmade for somebody like me
[37.00 → 42.00]  Come on now follow my lead, I may be crazy don't mind me
[42.00 → 47.00]  Say boy let's not talk too much, grab on my waist and put that body on me
[47.00 → 51.00]  Come on now follow my lead, come come on now follow my lead
[53.00 → 55.00]  I'm in love with the shape of you
[55.00 → 57.00]  We push and pull like a magnet
[57.00 → 60.00]  Although my heart is falling too
[60.00 → 62.00]  I'm in love with your body

#Instrument Identification

In [ ]:
import torch
import torchaudio
import torch.nn.functional as F
from transformers import AutoFeatureExtractor, AutoModelForAudioClassification


In [ ]:
MODEL_NAME = "Bhaveen/epoch_musical_instruments_identification_2"

feature_extractor = AutoFeatureExtractor.from_pretrained(MODEL_NAME)
model = AutoModelForAudioClassification.from_pretrained(MODEL_NAME)
model.eval()


In [ ]:
AUDIO_PATH = "/content/separated/htdemucs_ft/Edd_Sheeran/no_vocals.wav"

audio, sr = torchaudio.load(AUDIO_PATH)

# Convert to mono
audio = audio.mean(dim=0, keepdim=True)

# Resample to 16kHz (model requirement)
if sr != 16000:
    resampler = torchaudio.transforms.Resample(sr, 16000)
    audio = resampler(audio)


In [ ]:
def chunk_audio(audio, sample_rate=16000, chunk_sec=3.0, overlap=0.5):
    chunk_size = int(chunk_sec * sample_rate)
    hop = int(chunk_size * (1 - overlap))

    chunks = []
    for start in range(0, audio.shape[-1] - chunk_size + 1, hop):
        chunk = audio[..., start:start + chunk_size]
        chunks.append(chunk)

    return chunks


In [ ]:
def is_active(chunk, energy_threshold=1e-4):
    return chunk.pow(2).mean() > energy_threshold


In [ ]:
chunks = chunk_audio(audio)
all_probs = []

for chunk in chunks:
    if not is_active(chunk):
        continue

    inputs = feature_extractor(
        chunk.squeeze().numpy(),
        sampling_rate=16000,
        return_tensors="pt"
    )

    with torch.no_grad():
        logits = model(**inputs).logits
        probs = torch.sigmoid(logits)[0]  # ✅ SIGMOID (multi-label)

    all_probs.append(probs)

assert len(all_probs) > 0, "No active audio chunks detected!"


In [ ]:
all_probs = torch.stack(all_probs)  # [chunks, instruments]

instrument_scores = {
    model.config.id2label[i]: float(all_probs[:, i].max())
    for i in range(all_probs.shape[1])
}


In [ ]:
# Optional: remove extremely weak detections
instrument_scores = {
    k: round(v, 3)
    for k, v in instrument_scores.items()
    if v > 0.05
}


In [ ]:
print(instrument_scores)


#Graphical Representation of instrument

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

labels = list(instrument_scores.keys())
values = list(instrument_scores.values())

angles = np.linspace(0, 2 * np.pi, len(labels), endpoint=False)
values += values[:1]
angles = np.append(angles, angles[0])

plt.figure(figsize=(6, 6))
ax = plt.subplot(111, polar=True)
ax.plot(angles, values)
ax.fill(angles, values, alpha=0.25)

ax.set_thetagrids(angles[:-1] * 180 / np.pi, labels)
ax.set_ylim(0, 1)
ax.set_title("Instrument Distribution Radar")
plt.show()
